# Process PathoCell Data into CellWhisperer Format

This notebook processes the PathoCell dataset from HDF5 format to AnnData format.
It extracts the first TMA and formats it similar to the lymphoma_cosmx_small dataset.


In [ ]:
import h5py
import numpy as np
import pandas as pd
import anndata
from pathlib import Path
import json
import logging
from PIL import Image
import re

logging.basicConfig(level=logging.INFO)

In [ ]:
# Get paths from snakemake
data_dir = Path(snakemake.input.data_dir)
output_adata_path = Path(snakemake.output.adata)
output_image_path = Path(snakemake.output.image)
output_metadata_path = Path(snakemake.output.metadata)
ct_mapping_fine_path = Path(snakemake.input.ct_mapping_fine)
ct_mapping_coarse_path = Path(snakemake.input.ct_mapping_coarse)
patch_level = bool(getattr(snakemake.params, "patch_level", False))

logging.info(f"Reading PathoCell data from {data_dir}")
logging.info(f"Will save processed data to {output_adata_path}")
logging.info(f"Patch-level mode: {patch_level}")

In [ ]:
# Load specified TMA from Snakemake input
pathocell_hdf_dir = data_dir / "pathocell_hdf"
if not pathocell_hdf_dir.exists():
    raise FileNotFoundError(f"pathocell_hdf directory not found in {data_dir}")

hdf_files = list(pathocell_hdf_dir.glob("*.hdf"))
if not hdf_files:
    raise FileNotFoundError(f"No .hdf files found in {pathocell_hdf_dir}")

# Use the first .hdf file (TMA)
hdf_file = Path(snakemake.input.hdf_file)
logging.info(f"Processing {hdf_file.name}")

with h5py.File(hdf_file, "r") as f:
    # Explore the structure
    logging.info(f"HDF5 keys: {list(f.keys())}")

    # Extract image
    image = f["img"][:]
    assert len(image.shape) == 3
    if image.shape[0] == 3:
        image = np.transpose(image, (1, 2, 0))
    # Load mask data (instance segmentation)
    mask = f["gt_inst"][:]

    # Load fine/coarse label maps
    label_map_fine = f["gt_ct"][0]
    label_map_coarse = f["gt_ct_coarse"][0]

# Parse fine/coarse class names from mapping files
ct_mapping_fine_text = ct_mapping_fine_path.read_text()
ct_mapping_coarse_text = ct_mapping_coarse_path.read_text()
pairs_f = re.findall(r'(\d+)\s*:\s*"([^"]+)"', ct_mapping_fine_text)
pairs_c = re.findall(r"(\d+)\s*:\s*'([^']+)'", ct_mapping_coarse_text)
pairs_f = sorted(((int(k), v) for k, v in pairs_f), key=lambda x: x[0])
pairs_c = sorted(((int(k), v) for k, v in pairs_c), key=lambda x: x[0])
class_names_fine = [v for _, v in pairs_f]
class_names_coarse = [v for _, v in pairs_c]

In [ ]:
# Extract cell centroids and labels
unique_cell_ids = np.unique(mask)
unique_cell_ids = unique_cell_ids[unique_cell_ids > 0]
logging.info(f"Found {len(unique_cell_ids)} cells")

cell_data = []
for cell_id in unique_cell_ids:
    cell_mask = mask == cell_id
    ys, xs = np.where(cell_mask[0])
    if len(ys) == 0:
        continue
    y_pixel = ys.mean()
    x_pixel = xs.mean()
    fine_idx = (
        int(label_map_fine[int(ys[0]), int(xs[0])])
        if label_map_fine is not None
        else None
    )
    coarse_idx = (
        int(label_map_coarse[int(ys[0]), int(xs[0])])
        if label_map_coarse is not None
        else None
    )
    cell_data.append(
        {
            "cell_id": str(cell_id),
            "x_pixel": x_pixel,
            "y_pixel": y_pixel,
            "cell_type": class_names_fine[fine_idx] if fine_idx is not None else None,
            "cell_type_coarse": (
                class_names_coarse[coarse_idx] if coarse_idx is not None else None
            ),
        }
    )

logging.info(f"Processed {len(cell_data)} valid cells")

In [ ]:
# Cell dataframe and array coordinates
cell_df = pd.DataFrame(cell_data)
patch_size = 224
cell_df["x_array"] = (cell_df["x_pixel"] / patch_size).astype(int)
cell_df["y_array"] = (cell_df["y_pixel"] / patch_size).astype(int)
cell_df.index = [f"cell_{i}" for i in range(len(cell_df))]

In [ ]:

# Drop the cells with max x_array and max y_array (as they will be out of bounds partially)
max_x_array = cell_df["x_array"].max()
max_y_array = cell_df["y_array"].max()
cell_df = cell_df[
    (cell_df["x_array"] < max_x_array) & (cell_df["y_array"] < max_y_array)
]

In [ ]:
# Build AnnData (cell- or patch-level)
if not patch_level:
    adata = anndata.AnnData(obs=cell_df)
    adata.obsm["spatial"] = cell_df[["x_pixel", "y_pixel"]].values
    adata.obsm["X_spatial"] = adata.obsm["spatial"]
    adata.obs["cell_type"] = cell_df["cell_type"]
    adata.obs["cell_type_coarse"] = cell_df["cell_type_coarse"]

    # Build one-hot counts for fine/coarse classes
    ct_to_idx_f = {ct: i for i, ct in enumerate(class_names_fine)}
    counts_fine = np.zeros((adata.n_obs, len(class_names_fine)), dtype=int)
    for i, ct in enumerate(adata.obs['cell_type'].values):
        counts_fine[i, ct_to_idx_f[ct]] = 1
    adata.obsm['cell_type_counts'] = pd.DataFrame(
        counts_fine, index=adata.obs_names, columns=class_names_fine
    )

    if 'cell_type_coarse' in adata.obs and adata.obs['cell_type_coarse'].notna().any():
        ct_to_idx_c = {ct: i for i, ct in enumerate(class_names_coarse)}
        counts_coarse = np.zeros((adata.n_obs, len(class_names_coarse)), dtype=int)
        for i, ct in enumerate(adata.obs['cell_type_coarse'].fillna('').values):
            if ct in ct_to_idx_c:
                counts_coarse[i, ct_to_idx_c[ct]] = 1
        adata.obsm['cell_type_counts_coarse'] = pd.DataFrame(
            counts_coarse, index=adata.obs_names, columns=class_names_coarse
        )
else:
    grouped = cell_df.groupby(["x_array", "y_array"])
    patch_rows = []
    counts_fine = np.zeros((len(grouped), len(class_names_fine)), dtype=int)
    counts_coarse = np.zeros((len(grouped), len(class_names_coarse)), dtype=int)
    ct_to_idx_f = {ct: i for i, ct in enumerate(class_names_fine)}
    ct_to_idx_c = {ct: i for i, ct in enumerate(class_names_coarse)}
    for pidx, ((xa, ya), df_grp) in enumerate(grouped):
        n = len(df_grp)
        x_center = (xa + 0.5) * patch_size
        y_center = (ya + 0.5) * patch_size
        for ct, cnt in df_grp["cell_type"].value_counts().items():
            counts_fine[pidx, ct_to_idx_f[ct]] = int(cnt)
        for ct, cnt in df_grp["cell_type_coarse"].dropna().value_counts().items():
            counts_coarse[pidx, ct_to_idx_c[ct]] = int(cnt)
        dom_f = class_names_fine[int(counts_fine[pidx].argmax())]
        dom_c = (
            class_names_coarse[int(counts_coarse[pidx].argmax())]
            if counts_coarse[pidx].sum() > 0
            else None
        )
        patch_rows.append(
            {
                "x_array": int(xa),
                "y_array": int(ya),
                "x_pixel": float(x_center),
                "y_pixel": float(y_center),
                "n_cells": int(n),
                "cell_type": dom_f,
                "cell_type_coarse": dom_c,
            }
        )
    patch_df = pd.DataFrame(patch_rows)
    patch_df.index = [f"patch_{i}" for i in range(len(patch_df))]
    counts_fine_df = pd.DataFrame(
        counts_fine, index=patch_df.index, columns=class_names_fine
    )
    counts_coarse_df = pd.DataFrame(
        counts_coarse, index=patch_df.index, columns=class_names_coarse
    )
    adata = anndata.AnnData(obs=patch_df)
    adata.obsm["spatial"] = patch_df[["x_pixel", "y_pixel"]].values
    adata.obsm["X_spatial"] = adata.obsm["spatial"]
    adata.obsm["cell_type_counts"] = counts_fine_df
    adata.obsm["cell_type_counts_coarse"] = counts_coarse_df
    adata.obs["cell_type"] = patch_df["cell_type"]
    adata.obs["cell_type_coarse"] = patch_df["cell_type_coarse"]

In [ ]:
# Spatial metadata
library_id = "pathocell_tma1"
adata.uns["spatial"] = {
    library_id: {
        "images": {},
        "scalefactors": {
            "tissue_hires_scalef": 1.0,
            "tissue_lowres_scalef": 0.1,
            "spot_diameter_fullres": patch_size,
        },
        "metadata": {"library_id": library_id},
    }
}
adata.uns["pixel_size"] = 0.5
adata.uns["dataset"] = "pathocell_tma1"
adata.uns["image_path"] = str(output_image_path)
adata.uns["class_names_fine"] = class_names_fine
adata.uns["class_names_coarse"] = class_names_coarse

logging.info(
    f"Created AnnData with {adata.n_obs} {'patches' if patch_level else 'cells'}"
)

In [ ]:
# Save the image as TIFF with .svs extension (compatible with OpenSlide)
if image.dtype != np.uint8:
    if image.max() <= 1.0:
        image = (image * 255).astype(np.uint8)
    elif image.dtype == np.uint16:
        image = (image / 256).astype(np.uint8)
    else:
        image = image.astype(np.uint8)
pil_image = Image.fromarray(image, mode="RGB")
pil_image.save(str(output_image_path), format="TIFF", compression="lzw")
logging.info(f"Saved image to {output_image_path}")

In [ ]:
# Save AnnData and metadata
output_adata_path.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(output_adata_path)
logging.info(f"Saved AnnData to {output_adata_path}")

metadata = {
    "dataset_name": "pathocell_tma1",
    "source_file": str(hdf_file),
    "n_cells_or_patches": int(adata.n_obs),
    "image_shape": list(image.shape),
    "mask_shape": list(mask.shape),
    "class_names_fine": class_names_fine,
    "class_names_coarse": class_names_coarse,
    "pixel_size_um": adata.uns["pixel_size"],
    "processing_info": {"patch_size": patch_size, "patch_level": patch_level},
}
output_metadata_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
logging.info(f"Saved metadata to {output_metadata_path}")
logging.info("Processing complete!")